In [ ]:
# Databricks notebook source

# =========================
# SETUP
# =========================
catalog = "cabpluse360_team2"
bronze_schema = "cabpulse360_bronze_team2"
silver_schema = "cabpulse360_silver_team2"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {silver_schema}")

print(f"Using catalog: {catalog}")
print(f"Using schema: {silver_schema}")

# =========================
# IMPORTS
# =========================
from pyspark.sql.functions import (
    col, trim, to_timestamp, to_date, when, row_number,
    hour, round as spark_round
)
from pyspark.sql.window import Window


df_raw = spark.table(f"{catalog}.{bronze_schema}.trips")

# Deduplicate columns first (in case bronze has both old and new names)
seen = set()
cols_to_keep = []
for c in df_raw.columns:
    if c not in seen:
        cols_to_keep.append(c)
        seen.add(c)
df_raw = df_raw.select(cols_to_keep)

# Drop truly rogue columns
rogue_cols = [c for c in df_raw.columns 
              if c in {"EXTRA", "MTA_TAX", "CONGESTION_SURCHARGE"}]
df_raw = df_raw.drop(*rogue_cols)

# Only rename PU/DO if the NEW names don't already exist
if "PU_LOCATION_ID" in df_raw.columns and "PICKUP_LOCATION_ID" not in df_raw.columns:
    df_raw = df_raw.withColumnRenamed("PU_LOCATION_ID", "PICKUP_LOCATION_ID")

if "DO_LOCATION_ID" in df_raw.columns and "DROPOFF_LOCATION_ID" not in df_raw.columns:
    df_raw = df_raw.withColumnRenamed("DO_LOCATION_ID", "DROPOFF_LOCATION_ID")

if "SURCHARGE" in df_raw.columns and "EXTRA" not in df_raw.columns:
    df_raw = df_raw.withColumnRenamed("SURCHARGE", "EXTRA")

#schema inforcement
df_trips = df_raw \
    .withColumn("VENDOR_ID", col("VENDOR_ID").cast("int")) \
    .withColumn("PICKUP_DATETIME", to_timestamp("PICKUP_DATETIME")) \
    .withColumn("DROPOFF_DATETIME", to_timestamp("DROPOFF_DATETIME")) \
    .withColumn("PASSENGER_COUNT", col("PASSENGER_COUNT").cast("int")) \
    .withColumn("TRIP_DISTANCE", col("TRIP_DISTANCE").cast("double")) \
    .withColumn("PICKUP_LOCATION_ID", col("PICKUP_LOCATION_ID").cast("int")) \
    .withColumn("DROPOFF_LOCATION_ID", col("DROPOFF_LOCATION_ID").cast("int")) \
    .withColumn("RATE_CODE_ID", col("RATE_CODE_ID").cast("int")) \
    .withColumn("PAYMENT_TYPE", col("PAYMENT_TYPE").cast("int")) \
    .withColumn("FARE_AMOUNT", col("FARE_AMOUNT").cast("double")) \
    .withColumn("EXTRA", col("EXTRA").cast("double")) \
    .withColumn("TIP_AMOUNT", col("TIP_AMOUNT").cast("double")) \
    .withColumn("TOLLS_AMOUNT", col("TOLLS_AMOUNT").cast("double")) \
    .withColumn("TOTAL_AMOUNT", col("TOTAL_AMOUNT").cast("double"))

# DEDUPLICATION
window_dedup = Window.partitionBy(
    "VENDOR_ID", "PICKUP_DATETIME", "DROPOFF_DATETIME",
    "PICKUP_LOCATION_ID", "DROPOFF_LOCATION_ID"
).orderBy("_loaded_at")

df_trips = df_trips \
    .withColumn("_row_num", row_number().over(window_dedup)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

# DERIVED COLUMNS
df_trips = df_trips \
    .withColumn("TRIP_DATE", to_date("PICKUP_DATETIME")) \
    .withColumn("PICKUP_HOUR", hour("PICKUP_DATETIME")) \
    .withColumn("TRIP_DURATION_MIN",
        spark_round(
            (col("DROPOFF_DATETIME").cast("long") - col("PICKUP_DATETIME").cast("long")) / 60, 2
        )
    ) \
    .withColumn("TIME_BLOCK",
        when(col("PICKUP_HOUR").between(5, 8), "EARLY_MORNING")
        .when(col("PICKUP_HOUR").between(9, 11), "MORNING")
        .when(col("PICKUP_HOUR").between(12, 14), "AFTERNOON")
        .when(col("PICKUP_HOUR").between(15, 17), "EVENING")
        .when(col("PICKUP_HOUR").between(18, 20), "NIGHT")
        .otherwise("LATE_NIGHT")
    ) \
    .withColumn("FARE_CATEGORY",
        when(col("FARE_AMOUNT") <= 0, "ZERO_OR_NEGATIVE")
        .when(col("FARE_AMOUNT") <= 10, "LOW")
        .when(col("FARE_AMOUNT") <= 30, "MEDIUM")
        .when(col("FARE_AMOUNT") <= 60, "HIGH")
        .otherwise("PREMIUM")
    )

# WRITE
df_trips.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("trips")
    
print(f"silver.trips: {spark.table('trips').count()} rows")

# ============================================================
# SILVER TABLE 2: vendors
# ============================================================
df_vendors = spark.table(f"{catalog}.{bronze_schema}.vendors") \
    .withColumn("VENDOR_ID", col("VENDOR_ID").cast("int")) \
    .withColumn("VENDOR_NAME", trim("VENDOR_NAME")) \
    .dropDuplicates(["VENDOR_ID"])

df_vendors.write.mode("overwrite").saveAsTable("vendors")
print(f"silver.vendors: {spark.table('vendors').count()} rows")

# ============================================================
# SILVER TABLE 3: taxi_zones
# ============================================================
df_zones = spark.table(f"{catalog}.{bronze_schema}.taxi_zones") \
    .withColumnRenamed("ZONE_NAME", "ZONE") \
    .withColumn("LOCATION_ID", col("LOCATION_ID").cast("int")) \
    .withColumn("BOROUGH", trim("BOROUGH")) \
    .withColumn("ZONE", trim("ZONE")) \
    .dropDuplicates(["LOCATION_ID"])

df_zones.write.mode("overwrite").saveAsTable("taxi_zones")
print(f"silver.taxi_zones: {spark.table('taxi_zones').count()} rows")

# ============================================================
# SILVER TABLE 4: rate_codes
# ============================================================
df_rates = spark.table(f"{catalog}.{bronze_schema}.rate_codes") \
    .withColumn("RATE_CODE_ID", col("RATE_CODE_ID").cast("int")) \
    .withColumn("RATE_CODE_NAME", trim("RATE_CODE_NAME")) \
    .dropDuplicates(["RATE_CODE_ID"])

df_rates.write.mode("overwrite").saveAsTable("rate_codes")
print(f"silver.rate_codes: {spark.table('rate_codes').count()} rows")

# ============================================================
# SILVER TABLE 5: payment_types
# ============================================================
df_payments = spark.table(f"{catalog}.{bronze_schema}.payment_types") \
    .withColumn("PAYMENT_TYPE_ID", col("PAYMENT_TYPE_ID").cast("int")) \
    .withColumn("PAYMENT_TYPE_NAME", trim("PAYMENT_TYPE_NAME")) \
    .dropDuplicates(["PAYMENT_TYPE_ID"])

df_payments.write.mode("overwrite").saveAsTable("payment_types")
print(f"silver.payment_types: {spark.table('payment_types').count()} rows")

# ============================================================
# SILVER TABLE 6: vendor_name_changes
# ============================================================
from pyspark.sql.functions import to_date

df_vchanges = spark.table(f"{catalog}.{bronze_schema}.vendor_name_changes") \
    .withColumn("VENDOR_ID", col("VENDOR_ID").cast("int")) \
    .withColumn("OLD_NAME", trim("OLD_NAME")) \
    .withColumn("NEW_NAME", trim("NEW_NAME")) \
    .withColumn("CHANGE_DATE", to_date("CHANGE_DATE")) \
    .dropDuplicates(["VENDOR_ID", "CHANGE_DATE"])

df_vchanges.write.mode("overwrite").saveAsTable("vendor_name_changes")
print(f"silver.vendor_name_changes: {spark.table('vendor_name_changes').count()} rows")

# =========================
# VERIFY
# =========================
print("=== Silver Layer Summary ===")
for t in ["trips", "vendors", "taxi_zones", "rate_codes", "payment_types", "vendor_name_changes"]:
    print(f"{t}: {spark.table(t).count()} rows")